In [1]:
!pip install -q crewai litellm

In [2]:
import os
from google.colab import userdata

# Fetch from Colab Secrets
gemini_key = userdata.get('GEMINI_API_KEY')

# Create Streamlit's hidden secrets file
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/secrets.toml", "w") as f:
    f.write(f'GEMINI_API_KEY = "{gemini_key}"\n')

In [3]:
%%writefile app.py
import os
import time
import pandas as pd
import streamlit as st
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

# ==========================================
# 🛑 PASTE YOUR ACTUAL GEMINI API KEY BELOW
# ==========================================
# Pull the key securely from Streamlit's configuration
HACKATHON_API_KEY = st.secrets["GEMINI_API_KEY"]
os.environ["GEMINI_API_KEY"] = HACKATHON_API_KEY

# 1. Page Configuration
st.set_page_config(page_title="FinGuard AI", page_icon="🏦", layout="wide", initial_sidebar_state="collapsed")

# Initialize Session State
if 'page' not in st.session_state:
    st.session_state.page = "Dashboard"
if 'audit_result' not in st.session_state:
    st.session_state.audit_result = None

def launch_workspace():
    st.session_state.page = "Agent Workspace"

# 2. Premium Minimalist CSS
st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');

    html, body, [class*="css"] { font-family: 'Inter', sans-serif; color: #1a1a1a; }
    .stApp { background-color: #ffffff; }
    #MainMenu {visibility: hidden;}
    header {visibility: hidden;}

    .eyebrow { font-size: 11px; font-weight: 600; letter-spacing: 1.5px; color: #777777; text-transform: uppercase; margin-bottom: 8px; }
    .top-nav { background-color: #ffffff; padding: 0px 0px 20px 0px; border-bottom: 1px solid #eaeaea; margin-bottom: 30px; }
    .nav-header-row { display: flex; align-items: center; }
    .nav-logo { font-size: 36px; margin-right: 15px; }
    .nav-title { font-size: 28px; font-weight: 700; margin: 0; color: #111111; letter-spacing: -0.8px;}
    .nav-subtitle { color: #888888; margin-left: auto; font-weight: 400; font-size: 14px;}

    div.row-widget.stRadio > div { display: flex; flex-direction: row; justify-content: flex-start; gap: 30px; padding: 0px 0px 10px 0px; border-bottom: 1px solid #eaeaea; margin-bottom: 30px; }
    .stTextInput>div>div>input, .stSelectbox>div>div>div, .stTextArea>div>div>textarea, .stNumberInput>div>div>input { border-radius: 6px; border: 1px solid #e0e0e0; background-color: #fcfcfc; }

    .stButton>button { border-radius: 6px; font-weight: 500; transition: all 0.2s ease; }

    /* Target primary buttons to make them red */
    .stButton>button[kind="primary"] { background-color: #ef4444 !important; border: 1px solid #ef4444 !important; color: white !important; }
    .stButton>button[kind="primary"]:hover { background-color: #dc2626 !important; border: 1px solid #dc2626 !important; }

    .status-card { border: 1px solid #eaeaea; border-radius: 8px; padding: 15px; background-color: #fafafa; margin-bottom: 10px; }
    .status-title { font-size: 13px; color: #666; font-weight: 600; margin-bottom: 8px; text-transform: uppercase; letter-spacing: 0.5px;}
    .status-value { font-size: 15px; font-weight: 500; display: flex; align-items: center; gap: 8px;}
    .dot-green { height: 10px; width: 10px; background-color: #10b981; border-radius: 50%; display: inline-block; box-shadow: 0 0 8px #10b98140;}
    .dot-gray { height: 10px; width: 10px; background-color: #9ca3af; border-radius: 50%; display: inline-block; }
    h1, h2, h3 { color: #111111; font-weight: 600; letter-spacing: -0.5px;}
    </style>
""", unsafe_allow_html=True)

# 3. Minimalist Header
st.markdown('''
    <div class="top-nav">
        <div class="eyebrow">Bank fee Auditor • Hackathon Build</div>
        <div class="nav-header-row">
            <span class="nav-logo">🏦</span>
            <h1 class="nav-title">Finguard AI</h1>
            <span class="nav-subtitle">Secure Workspace v1.2</span>
        </div>
    </div>
''', unsafe_allow_html=True)

# 4. Horizontal Navigation Bar
st.radio("Navigation", ["Dashboard", "Agent Workspace", "Audit History", "How to Use", "Documentation"], horizontal=True, label_visibility="collapsed", key="page")

# Core Logic & Tools
OFFICIAL_TARIFFS = {
    "Classic Savings": {"min_balance": 5000, "non_maintenance_fee": 250, "atm_charge": 25},
    "Premium Wealth": {"min_balance": 25000, "non_maintenance_fee": 500, "atm_charge": 0}
}

@tool("Tariff Verification Tool")
def verify_bank_fee(tier: str, fee_desc: str, amount: float, balance: float) -> str:
    """Verifies if a fee matches the bank's published tariff."""
    try:
        if tier not in OFFICIAL_TARIFFS:
            return f"SYSTEM ERROR: Unknown tier '{tier}'."
        rules = OFFICIAL_TARIFFS[tier]
        desc = str(fee_desc).lower()
        amount = float(amount)
        balance = float(balance)

        if "credit" in desc or "dividend" in desc or "refund" in desc:
            return "IGNORED: This is an incoming credit, not a bank fee."
        if "non-maintenance" in desc or "min balance" in desc:
            if balance >= rules["min_balance"]:
                return "FLAGGED BREACH: Minimum balance fee charged illegally. Maintained balance exceeds requirement."
            elif amount != rules["non_maintenance_fee"]:
                return f"FLAGGED BREACH: Incorrect tier fee applied. Tariff is Rs. {rules['non_maintenance_fee']}."
            return "VALID CHARGE: Fee applied correctly."
        if "atm" in desc:
            if rules["atm_charge"] == 0 and amount > 0:
                return "FLAGGED BREACH: Account features unlimited free ATM usage."
            elif amount != rules["atm_charge"]:
                return f"FLAGGED BREACH: Incorrect ATM fee. Standard rate is Rs. {rules['atm_charge']}."
            return "VALID CHARGE: ATM fee is accurate."
        return "NEUTRAL: Charge not classified under standard tariffs."
    except Exception as e:
        return "TOOL ERROR: Failed to parse transaction data."

# ==========================================
# PAGE ROUTING LOGIC
# ==========================================

if st.session_state.page == "Dashboard":
    col1, col2 = st.columns([2, 1])
    with col1:
        st.subheader("System Overview")
        st.write("Welcome to the  automated compliance dashboard. Monitor retail banking tariffs, identify erroneous fee deductions, and seamlessly draft reversal claims using deterministic AI tooling.")
        st.write("")
        # Made type="primary" to apply the red CSS styling
        st.button("Launch Agent Workspace →", type="primary", on_click=launch_workspace)

    with col2:
        st.markdown('''
            <div class="status-card">
                <div class="status-title">Core Banking API</div>
                <div class="status-value"><span class="dot-green"></span> Synchronized (9ms)</div>
            </div>
            <div class="status-card">
                <div class="status-title">Gemini LLM Orchestrator</div>
                <div class="status-value"><span class="dot-green"></span> Online & Authenticated</div>
            </div>
            <div class="status-card">
                <div class="status-title">CrewAI Multi-Agent Engine</div>
                <div class="status-value"><span class="dot-green"></span> Standby</div>
            </div>
            <div class="status-card">
                <div class="status-title">Automated Dispatch</div>
                <div class="status-value"><span class="dot-gray"></span> Locked (Human Auth Required)</div>
            </div>
        ''', unsafe_allow_html=True)

    st.markdown("---")
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Total Audited (30d)", "1,248", "↑ 12%")
    m2.metric("Breaches Flagged", "42", "↓ 5%")
    m3.metric("Funds Reversed", "Rs. 18,500", "↑ Rs. 2k")
    m4.metric("AI Grounding Accuracy", "99.9%", "Stable")

elif st.session_state.page == "Agent Workspace":
    col1, col2 = st.columns([1, 1], gap="large")

    with col1:
        st.subheader("Input Statement")
        account_tier = st.selectbox("Account Tier", list(OFFICIAL_TARIFFS.keys()))
        monthly_balance = st.number_input("Monthly Average Balance (Rs.)", value=7450.0, step=100.0)

        default_txns = (
            "2026-10-05: Debit - Non-maintenance of Minimum Balance Fee: Rs. 500.00 (Reference: CHG-9921)\n"
            "2026-10-12: Credit - Dividend Payment from Tata Motors: Rs. 350.00 (Unrelated Credit)\n"
            "2026-10-18: Debit - Out-of-network ATM Withdrawal Surcharge: Rs. 25.00"
        )
        transactions = st.text_area("Paste Statement Transactions", value=default_txns, height=200)
        run_button = st.button("Execute Compliance Audit", use_container_width=True)

    with col2:
        st.subheader("Agent Output")

        if run_button:
            if HACKATHON_API_KEY == "PASTE_YOUR_REAL_KEY_HERE":
                st.error("⚠️ Developer Error: You forgot to paste your API key!")
            else:
                with st.status("Initializing Multi-Agent System...", expanded=True) as status:
                    # Model cascade logic: automatically fallback to lower tiers if the higher one fails
                    models_to_try = [
                        "gemini/gemini-3.5-flash-lite",
                        "gemini/gemini-3.6-flash",
                        "gemini/gemini-3.8-flash"
                    ]

                    success = False
                    last_error = None

                    for model_tier in models_to_try:
                        try:
                            st.write(f"Attempting execution via `{model_tier}`...")
                            gemini_llm = LLM(model=model_tier, api_key=HACKATHON_API_KEY, temperature=0.1)

                            auditor = Agent(role="Deterministic Bank Fee Auditor", goal="Pass every transaction to the Tariff Verification Tool.", backstory="Strict compliance bot. You do not do math. You only pass numbers to your tool.", tools=[verify_bank_fee], llm=gemini_llm)
                            claims_agent = Agent(role="Reversal Claims Specialist", goal="Draft a formal reversal email based ONLY on 'FLAGGED BREACH' tool outputs.", backstory="Professional banking operative. You write clean dispute emails.", llm=gemini_llm)

                            audit_task = Task(description=f"Analyze account '{account_tier}' with MAB Rs. {monthly_balance}.\nTransactions:\n{transactions}\nACTION: Feed every debit to the Verification Tool. Output a clear table of results.", expected_output="Markdown table showing Date, Description, Amount, and Tool Verdict.", agent=auditor)
                            draft_task = Task(description="If the audit found a 'FLAGGED BREACH', draft ONE concise reversal email to Bank Support.", expected_output="A single ready-to-dispatch reversal email.", agent=claims_agent)

                            crew = Crew(agents=[auditor, claims_agent], tasks=[audit_task, draft_task], process=Process.sequential)
                            st.session_state.audit_result = str(crew.kickoff())

                            status.update(label=f"Audit Complete! (Powered by {model_tier.split('/')[-1]})", state="complete", expanded=False)
                            success = True
                            break # Exit the loop gracefully once a model successfully executes

                        except Exception as e:
                            last_error = str(e)
                            st.warning(f"⚠️ `{model_tier}` unavailable or rate-limited. Falling back to next available tier...")
                            time.sleep(1) # Brief pause before retrying

                    if not success:
                        status.update(label="Audit Failed: All Model Tiers Exhausted", state="error")
                        st.error(f"🚨 System Error: {last_error}")

        if st.session_state.audit_result:
            st.success("Workflow Executed Successfully")
            st.markdown(st.session_state.audit_result)

            st.markdown("---")
            st.write("**Final Human Authorization**")
            st.caption("Regulatory compliance requires human sign-off before dispatching to the bank's API.")

            confirm_tx = st.checkbox("I confirm the disputed transaction details and tariff rules are correct.")
            confirm_bal = st.checkbox("I verify that unrelated credits were safely ignored.")

            if st.button("Authorize & Dispatch", type="primary", disabled=not (confirm_tx and confirm_bal)):
                with st.spinner("Dispatching payload..."):
                    time.sleep(1.5)
                st.success("Payload securely transmitted. (Ticket ID: TKT-99421)")
                st.toast("Transmission Complete", icon="🔒")

elif st.session_state.page == "Audit History":
    st.subheader("Secure Audit History")
    mock_data = pd.DataFrame({
        "Reference ID": ["TKT-99422", "TKT-99421", "TKT-99405", "TKT-99399", "TKT-99380", "TKT-99371", "TKT-99350", "TKT-99312"],
        "Audit Date": ["2026-10-22", "2026-10-22", "2026-10-21", "2026-10-21", "2026-10-20", "2026-10-20", "2026-10-19", "2026-10-18"],
        "Account Tier": ["Classic Savings", "Premium Wealth", "Classic Savings", "Premium Wealth", "Classic Savings", "Classic Savings", "Premium Wealth", "Classic Savings"],
        "Status": ["Action Required", "Compliant", "Action Required", "Compliant", "Action Required", "Compliant", "Compliant", "Action Required"],
        "Anomalies Found": [1, 0, 2, 0, 1, 0, 0, 3],
        "Human Approval": ["Pending", "N/A", "Signed-off", "N/A", "Signed-off", "N/A", "N/A", "Signed-off"]
    })
    st.dataframe(mock_data, use_container_width=True, hide_index=True)

elif st.session_state.page == "How to Use":
    st.subheader("Operator Instructions")
    st.markdown("""
    Follow these steps to conduct an automated compliance audit:

    1. **Navigate to the Agent Workspace**: Use the tabs above or the 'Launch' button on the dashboard.
    2. **Select Account Tier**: Choose the banking tier the customer belongs to (e.g., 'Classic Savings'). The app will dynamically pull the hardcoded master rules for that specific tier.
    3. **Input MAB**: Enter the Monthly Average Balance maintained by the user.
    4. **Provide Transactions**: Paste the raw text of the user's transaction statement.
    5. **Execute Audit**: Click the execution button.
       * *Agent 1 (Auditor)* reads the statement, avoids doing any math itself, and routes all numbers strictly through the Python deterministic tool.
       * *Agent 2 (Claims Specialist)* reads the tool's findings and drafts a professional reversal ticket.
    6. **Authorize**: Review the AI's findings. The system will physically lock the dispatch button until a human verifies the checkboxes.
    """)

elif st.session_state.page == "Documentation":
    st.subheader("Architecture")
    st.write("""
    Built for the ** Agentic AI Hackathon**.

    **Tech Stack:**
    * **Frontend:** Streamlit
    * **Orchestration:** CrewAI (Sequential Process)
    * **LLM:** Google Gemini Flash
    * **Safety Constraint:** Deterministic Python Tooling (Zero-Math AI Policy) + Mandatory Human Checkpoint.

    Designed to eliminate LLM hallucination in financial mathematics by forcing the AI to act exclusively as a natural-language bridge to hardcoded logic.
    """)

Overwriting app.py


In [5]:
# Force delete the old hidden config file to prevent conflicts
!rm -f /root/.config/ngrok/ngrok.yml

from pyngrok import ngrok
from google.colab import userdata

# Pull the key securely from Colab Secrets
ngrok_token = userdata.get('NGROK_AUTH_TOKEN')

ngrok.kill()
ngrok.set_auth_token(ngrok_token)
public_url = ngrok.connect(8501)
print("Here is your NEW Live Demo URL:", public_url)

Here is your NEW Live Demo URL: NgrokTunnel: "https://bonelike-upstate-napping.ngrok-free.dev" -> "http://localhost:8501"
